# Understanding Dual Values Well Enough to Document Them

**What this notebook is for:** helping you get inside the current dual-value workflow in Sage / Passagemath well enough to write a clear section for `linear_programming.rst`.

**What you should get out of it:** not just a doc patch idea, but a working mental model of:

- what a dual value means,
- where it lives in the current stack,
- why the backend matters,
- why the docs need careful wording.

**Format:** part runnable LP experiment in this notebook, part guided source-reading in the real `passagemath` repository.

**Center of gravity:** the perturbation experiment in the middle of the notebook. That is where the API call turns into an interpretation.

**How to use this notebook:** run it in order and jot down your own answers as you go. The places that ask for an explanation are where the understanding usually clicks.


In [ ]:
# Setup check -- run this first.
# If this import fails, switch to a kernel/environment where Sage works.

from sage.numerical.mip import MixedIntegerLinearProgram
print("MixedIntegerLinearProgram imported OK.")


---
## Section 1: Start with your intuition

Before touching the source, make a prediction.

Suppose you build and solve a linear program. Where would you expect to find dual values?

Possible guesses:

- directly on `MixedIntegerLinearProgram`,
- on constraint objects,
- on the backend,
- nowhere at all.

It is useful to surface your assumptions first and then watch the code sharpen them.


In [ ]:
my_initial_guess = {
    "where_i_expect_duals": "",
    "whether_i_expect_a_high_level_api": "",
    "whether_i_expect_this_for_generic_mips": "",
    "plain_english_meaning_of_a_dual_value": "",
}
my_initial_guess


---
## Section 2: Solve a tiny LP first

Start with the object you are trying to understand, then come back to the docs.

This is a tiny linear program:

- maximize `3x + 2y`
- subject to `x + y <= 4`
- and `x <= 2`
- and `y <= 3`
- with `x, y >= 0`

This example gives three useful cases in one small model:

- a shared resource constraint,
- an individual cap that still matters,
- and a cap that may turn out not to matter at the optimum.

Run it. Then look at the optimal solution and ask yourself what an extra unit of each resource might be worth.


In [ ]:
p = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
x = p.new_variable(nonnegative=True)

p.set_objective(3*x[0] + 2*x[1])
p.add_constraint(x[0] + x[1] <= 4)
p.add_constraint(x[0] <= 2)
p.add_constraint(x[1] <= 3)

# We are interested in LP dual information, so keep the solve in simplex mode.
p.solver_parameter('simplex_or_intopt', 'simplex_only')

opt = p.solve()
vals = p.get_values(x)

print("objective:", opt)
print("variable values:", vals)


Take a minute here.

Without looking anything up yet, answer:

1. Which of the three constraints do you think are binding at the optimum?
2. If you were allowed one extra unit on the shared-resource inequality `x + y <= 4`, do you expect the objective to go up?
3. What about one extra unit on `x <= 2`?
4. What about one extra unit on `y <= 3`?

This is a quick intuition check before the software gives you the answer.


In [ ]:
pre_dual_intuition = {
    "which_constraint_feels_more_valuable": "",
    "why": "",
    "my_guess_for_shadow_prices": "",
}
pre_dual_intuition


---
## Section 3: Go find the numbers

Now ask the software where the dual values actually live.

What matters here is the boundary between frontend and backend:

- you built the model through the frontend,
- but dual values may not live on the frontend object itself.

Retrieve the row duals and see what layer of the system they come from.


In [ ]:
b = p.get_backend()

print("backend type:", type(b))
print("row dual 0:", b.get_row_dual(0))
print("row dual 1:", b.get_row_dual(1))
print("row dual 2:", b.get_row_dual(2))


This is the first interesting distinction.

You did not call a generic high-level method like `p.get_dual_values()`.
You solved the model, got the backend, and then asked the backend for row duals.

That distinction should shape how the docs describe the feature.


In [ ]:
frontend_backend_takeaway = """
Write 2-4 sentences.

What did this teach you about where dual values live in the current stack?
Why does that matter for how a docs section should be framed?
"""
print(frontend_backend_takeaway)


---
## Section 4: Figure out which row is which

A reported number gets much more interesting once you know what it belongs to.

The practical question is simple:

- does row `0` correspond to the first constraint added?
- does row `1` correspond to the second?
- does row `2` correspond to the third?

Use this tiny model to pin that down.


In [ ]:
row_mapping_notes = {
    "what_row_0_means": "",
    "what_row_1_means": "",
    "what_row_2_means": "",
    "how_i_know": "",
}
row_mapping_notes


---
## Section 5: Verify the meaning, not just the API

This is the conceptual center of the notebook.

This experiment is where the numbers start to feel real.

If `get_row_dual(0)` says a constraint is worth some amount, test it.

Increase that constraint's right-hand side by `1`, solve again, and compare the new objective value.

That is the step that turns an API call into a shadow-price interpretation.

Once this part makes sense, the later documentation choices usually become much easier.


In [ ]:
p2 = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
y = p2.new_variable(nonnegative=True)

p2.set_objective(3*y[0] + 2*y[1])
p2.add_constraint(y[0] + y[1] <= 5)   # first RHS increased by 1
p2.add_constraint(y[0] <= 2)
p2.add_constraint(y[1] <= 3)
p2.solver_parameter('simplex_or_intopt', 'simplex_only')

opt2 = p2.solve()
vals2 = p2.get_values(y)
b2 = p2.get_backend()

print("new objective:", opt2)
print("new variable values:", vals2)
print("new row dual 0:", b2.get_row_dual(0))
print("new row dual 1:", b2.get_row_dual(1))
print("new row dual 2:", b2.get_row_dual(2))


In [ ]:
shadow_price_check = {
    "original_objective": opt,
    "new_objective": opt2,
    "objective_change": opt2 - opt,
    "original_row_dual_0": b.get_row_dual(0),
}
shadow_price_check


Now put the result into words.

Good target sentence shape:

- "The dual value for the first constraint is ... , which means ..."

If the objective change does not match what you expected, say why you think that is.


In [ ]:
shadow_price_explanation = """
Write 3-5 sentences explaining what you learned from the perturbation experiment.
"""
print(shadow_price_explanation)


---
## Section 6: Why the docs need careful wording

At this point you know two things:

- there is a real dual-value workflow,
- but it is accessed through the backend.

Now add the mathematical caution:

- dual values are natural LP sensitivity information,
- and they are easy to overstate if they are presented as a generic MIP feature.

Keep that in mind as you move into the source reading and the final synthesis.


---
## Section 7: Go read the source with better questions

Now switch to the real `passagemath` repository.

At this point the source reading usually feels different: you know what behavior you are trying to account for.

Search for the relevant implementation details:

```sh
rg -n 'get_row_dual|simplex_or_intopt|dual' src
```

And locate the docs target:

```sh
rg --files | rg 'linear_programming\.rst$'
```

The goal in this pass is to answer a focused set of questions about implementation and scope.


In [ ]:
source_reading_notes = {
    "where_linear_programming_rst_lives": "",
    "where_get_row_dual_is_implemented": "",
    "which_backend_path_i_verified": "",
    "what_solver_mode_requirement_i_found": "",
    "what_i_found_about_row_ordering": "",
    "what_claims_i_now_feel_safe_making": "",
}
source_reading_notes


Checkpoint.

At this point it is useful to check whether you can answer all of these in your own words:

1. What is the user-facing problem in the docs?
2. What is the actual workflow for getting row duals?
3. Why is this backend-specific enough to deserve careful framing?
4. What does one dual value mean in a concrete LP?

If one of those still feels fuzzy, it is worth another pass through either the tiny LP or the source.


---
## Section 8: Plan the smallest useful doc patch

With that background, the docs addition can stay small. A good first patch is probably:

- one short subsection,
- one worked example,
- one note or warning block.

This patch does not need to take on the broader interface debate.
It can stay focused on documenting an existing, useful path accurately.


---
## Section 9: Synthesize the big picture

Before drafting prose, pull the whole picture together.

A strong docs section here depends on seeing three things at once:

- the optimization meaning of the number,
- the fact that the current workflow goes through the backend,
- the scope limits on how broadly that should be presented.

If you can explain those three together, the draft usually comes out in the right register.


In [ ]:
big_picture_synthesis = """
Write 4-6 sentences for a future contributor.

Explain:
- what a dual value means in the LP you ran,
- why the current retrieval path goes through the backend,
- and why that should be documented as a backend-supported LP workflow rather than a broad MIP feature.
"""
print(big_picture_synthesis)


---
## Section 10: Draft the docs text

Draft a short section with only claims you verified.

Use this filter for every sentence:

- did I verify this in code?
- did I verify this experimentally?
- or am I only repeating something that sounds plausible?

That keeps the section technically grounded.


In [ ]:
rst_draft = r'''
.. Write your candidate RST section here.
'''
print(rst_draft)


---
## Section 11: Review it like a maintainer

Read your draft and ask:

1. Does this document current behavior rather than argue for a roadmap?
2. Is it obvious that the workflow goes through the backend?
3. Would a new reader understand what the number means, not just how to print it?
4. Have I implied broader support than I actually verified?

If one answer is "no", revise the draft while the experiment and source reading are still fresh.


---
## Summary

If you worked through this notebook, you should now know more than "how to call `get_row_dual(i)`".

You should understand:

- where the current dual-value path lives,
- what a shadow price means in a concrete LP,
- why the backend boundary matters for documentation,
- how to write a small doc contribution that is useful without overstating support,
- and how to investigate similar backend-exposed features elsewhere in the library.

